# NethoBench quickstart

This notebook uses the small dataset bundled with the repository. The goal is simple: load one prediction file for each benchmark mode, run the scorer, and inspect the main numbers.

Run it from a fresh checkout after installing the package with `pip install -e .`. If the notebook kernel was started somewhere other than the repository root, the setup cell below will still find the data folder.

## Setup

The imports are the same ones you would use in a script. The helper functions keep the output tables short so the notebook stays readable.

In [ ]:
from pathlib import Path

import pandas as pd

from nethobench import (
    compute_cross_scores,
    compute_etho_scores,
    compute_fidelity_scores,
    compute_neuro_scores,
)


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError("Could not find the repository root with pyproject.toml and data/.")


ROOT = find_repo_root()
DATA = ROOT / "data"
OUT = ROOT / "outputs" / "quickstart"
OUT.mkdir(parents=True, exist_ok=True)


def score_table(scores: dict, keys: list[str]) -> pd.DataFrame:
    rows = []
    for key in keys:
        value = scores.get(key, float("nan"))
        rows.append({"score": key, "value": float(value) if pd.notna(value) else value})
    return pd.DataFrame(rows)


ROOT

## Data files

The demo uses one neural prediction, one behavioral prediction, and one multimodal prediction. Other files in `data/` follow the same schema.

In [ ]:
neural_gt = DATA / "neural" / "gt" / "data-clean-300-1140-ba4.csv"
neural_pred = DATA / "neural" / "predictions" / "netho-seq-sl300-ba4-data100-seed102-epoch-10-predictions.csv"

behavior_gt = DATA / "behavioural" / "gt.parquet"
behavior_pred = DATA / "behavioural" / "predictions" / "sequifier-behav-seq-real-2-last-100-predictions.csv"

cross_gt = DATA / "cross" / "gt" / "cross-gt-behavior-neuro.csv"
cross_pred = DATA / "cross" / "predictions" / "sequifier-cross-noisy-behavior-neuro-last-100.csv"

for label, file_path in {
    "neural_gt": neural_gt,
    "neural_pred": neural_pred,
    "behavior_gt": behavior_gt,
    "behavior_pred": behavior_pred,
    "cross_gt": cross_gt,
    "cross_pred": cross_pred,
}.items():
    print(f"{label:13s} {file_path.relative_to(ROOT)}")
    assert file_path.exists(), file_path

## Neural and fidelity scores

`neuro-scores` measures structural realism. `fidelity-scores` is the direct trace-alignment family kept separate from the neural composite.

In [ ]:
neuro_scores = compute_neuro_scores(neural_pred, neural_gt)
fidelity_scores = compute_fidelity_scores(neural_pred, neural_gt)

score_table(
    neuro_scores,
    [
        "family_distribution",
        "family_temporal_spectral",
        "family_relational",
        "family_geometry",
        "family_state_dynamics",
        "FINAL_COMPOSITE_SCORE",
    ],
)

In [ ]:
score_table(fidelity_scores, ["Error_score", "MI_score", "FIDELITY_SCORE"])

## Behavioral scores

The behavioral scorer accepts either a single file or a directory of `.csv`/`.parquet` files. Here we use one prediction file so the result is easy to compare with the terminal command in the README.

In [ ]:
behavior_scores, per_sequence, per_sequence_mean, per_sequence_std = compute_etho_scores(
    behavior_gt,
    behavior_pred,
)

score_table(
    behavior_scores,
    [
        "position_kl_score",
        "velocity_score",
        "direction_score",
        "syllable_score",
        "trajectory_shape_score",
        "composite_score",
    ],
)

## Cross-modal scores

The multimodal file contains both neural columns and behavioral `_X`/`_Y` columns. With the default column names, no config file is needed.

In [ ]:
cross_result = compute_cross_scores(cross_pred, cross_gt, None)

score_table(
    {
        "neuro_composite": cross_result["neuro_composite"],
        "behavior_composite": cross_result["etho_composite"],
        "cross_composite": cross_result["cross_composite"],
        "final_composite": cross_result["composite"],
    },
    ["neuro_composite", "behavior_composite", "cross_composite", "final_composite"],
)

## What to change next

- Swap in another prediction file from `data/neural/predictions/` or `data/behavioural/predictions/`.
- Use `nethobench neuro-analysis`, `etho-analysis`, or `cross-analysis` from the terminal when you want figures as well as JSON scores.
- If your own files use different column names, add a small JSON config with `sequence_key`, `time_key`, `neuro_cols`, and `behavior_parts`.